In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import cv2
import pywt
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/Deepfake_Project"

INPUT_PATH = os.path.join(PROJECT_PATH, "aligned_faces")
OUTPUT_PATH = os.path.join(PROJECT_PATH, "dwt_fusion_images")

os.makedirs(os.path.join(OUTPUT_PATH, "original"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_PATH, "Deepfakes"), exist_ok=True)

In [ ]:
sample_video = os.listdir(os.path.join(INPUT_PATH, "original"))[0]
sample_frame = os.listdir(os.path.join(INPUT_PATH, "original", sample_video))[0]

image_path = os.path.join(INPUT_PATH, "original", sample_video, sample_frame)

image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

print(image.shape)

In [ ]:
R = image[:, :, 0]
G = image[:, :, 1]
B = image[:, :, 2]

In [ ]:
LL_R, (LH_R, HL_R, HH_R) = pywt.dwt2(R, 'haar')
LL_G, (LH_G, HL_G, HH_G) = pywt.dwt2(G, 'haar')
LL_B, (LH_B, HL_B, HH_B) = pywt.dwt2(B, 'haar')

In [ ]:
print("LL:", LL_R.shape)
print("LH:", LH_R.shape)
print("HL:", HL_R.shape)
print("HH:", HH_R.shape)

In [ ]:
def dwt_fusion_image(image):

    channels = []

    for c in range(3):

        channel = image[:, :, c]

        LL, (LH, HL, HH) = pywt.dwt2(channel, 'haar')

        # Weighted Frequency Fusion
        fused = (
            0.70 * LL +
            0.10 * np.abs(LH) +
            0.10 * np.abs(HL) +
            0.10 * np.abs(HH)
        )

        # Resize back to 224×224
        fused = cv2.resize(fused, (224, 224))

        # Normalize
        fused = cv2.normalize(
            fused,
            None,
            0,
            255,
            cv2.NORM_MINMAX
        ).astype(np.uint8)

        channels.append(fused)

    fused_image = cv2.merge(channels)

    return fused_image

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(8,8))

ax[0,0].imshow(LL_R, cmap='gray')
ax[0,0].set_title("LL")

ax[0,1].imshow(LH_R, cmap='gray')
ax[0,1].set_title("LH")

ax[1,0].imshow(HL_R, cmap='gray')
ax[1,0].set_title("HL")

ax[1,1].imshow(HH_R, cmap='gray')
ax[1,1].set_title("HH")

for a in ax.ravel():
    a.axis("off")

plt.show()

In [ ]:
def process_dwt_dataset(input_root, output_root):

    processed = 0

    classes = ["original", "Deepfakes"]

    for cls in classes:

        input_class = os.path.join(input_root, cls)
        output_class = os.path.join(output_root, cls)

        videos = sorted(os.listdir(input_class))

        for video in tqdm(videos, desc=f"{cls}"):

            input_video = os.path.join(input_class, video)
            output_video = os.path.join(output_class, video)

            os.makedirs(output_video, exist_ok=True)

            frames = sorted(os.listdir(input_video))

            for frame in frames:

                image_path = os.path.join(input_video, frame)

                image = cv2.imread(image_path)

                if image is None:
                    continue

                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

                fusion = dwt_fusion_image(image)

                save_path = os.path.join(output_video, frame)

                cv2.imwrite(
                    save_path,
                    cv2.cvtColor(fusion, cv2.COLOR_RGB2BGR)
                )

                processed += 1

    print("="*50)
    print("Completed")
    print("Processed :", processed)

In [ ]:
process_dwt_dataset(
    INPUT_PATH,
    OUTPUT_PATH
)

In [ ]:
sample_video = os.listdir(os.path.join(OUTPUT_PATH, "original"))[0]
sample_frame = os.listdir(os.path.join(OUTPUT_PATH, "original", sample_video))[0]

fusion_path = os.path.join(
    OUTPUT_PATH,
    "original",
    sample_video,
    sample_frame
)

fusion = cv2.imread(fusion_path)
fusion = cv2.cvtColor(fusion, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(5,5))
plt.imshow(fusion)
plt.title("Saved DWT Fusion Image")
plt.axis("off")
plt.show()